## Checking GPU Avalibality

In [ ]:
import tensorflow as tf

tf.__version__

In [ ]:
import time

print("GPUs:", tf.config.list_physical_devices("GPU"))

with tf.device("/GPU:0"):
    a = tf.random.normal([5000, 5000])
    b = tf.random.normal([5000, 5000])

    start = time.time()
    c = tf.matmul(a, b)
    c.numpy()

print("Time:", time.time() - start)

In [ ]:
# !nvidia-smi

### Used modules

In [ ]:
from src.metrics.der.diacritic_error_rate import DiacriticErrorRate
from src.callbacks.ce_der.case_ending_der import CaseEndingDERCallback
from src.callbacks.wer.word_error_rate import WordErrorRateCallback
from src.helpers.datasets_helper.dataset import Dataset

from src.util.vocab import DIAC_PAD_ID, DIAC_SPACE_ID, DIAC_VOCAB_LIST
from src.util.data import make_dataset, to_training_triplet, materialize_validation_set, encode_text
from src.models import MODEL_BUILDERS, decode_predictions
from src.callbacks.prediction_cache import PredictionCache

# Build dataset as well as model

## Variables

In [ ]:
# Consts

hidden_dim = 64
num_layers = 1
lr = 5e-3

model = "lstm"

epochs = 15
batch_size = 32

## Dataset

In [ ]:
# Select Dataset

ds = Dataset("sadeed_tashkeal")

In [ ]:
# Load

train_ds = ds.load_dataset("train", batch_size=batch_size)
test_ds = ds.load_dataset("test", batch_size=batch_size)

In [ ]:
# Pass the dataset through data pipeline

train_ds_full = make_dataset(train_ds)
test_ds_full  = make_dataset(test_ds)

# Configure case_ending_boost to makes the LOSS itself weight case-ending
train_ds = train_ds_full.map(
    lambda c, d, w, m: to_training_triplet(c, d, w, m, case_ending_boost=0.0)
).prefetch(tf.data.AUTOTUNE)

test_ds = test_ds_full.map(
    lambda c, d, w, m: to_training_triplet(c, d, w, m, case_ending_boost=0.0)
).prefetch(tf.data.AUTOTUNE)

x_val, y_val, case_ending_mask_val = materialize_validation_set(test_ds_full)

## Model

In [ ]:
model = MODEL_BUILDERS[model](
    hidden_dim=hidden_dim, 
    num_layers=num_layers, 
    bidirectional=True, 
    dropout=0.17, 
    embed_dim=126
)
    
model.summary()   

In [ ]:
# Metrics and callbacks

shared_cache = PredictionCache()

der = DiacriticErrorRate(pad_id=DIAC_PAD_ID)

model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=f'.cache/checkpoint.{model}.keras',
    monitor='val_der',
    mode='min',
    save_best_only=True
)

wer = WordErrorRateCallback(
    val_data=(x_val, y_val), 
    space_id=DIAC_SPACE_ID,
    pad_id=DIAC_PAD_ID,
    prediction_cache=shared_cache    
)

ce_der = CaseEndingDERCallback(
    val_data=(x_val, y_val, case_ending_mask_val), 
    pad_id=DIAC_PAD_ID,
    prediction_cache=shared_cache
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_der", 
    patience=3, 
    restore_best_weights=True,
    mode="min"
),

In [ ]:
# Compile

model.compile(
    optimizer=tf.keras.optimizers.Adam(lr),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
            der
        ],
    weighted_metrics=[],
)

## Train

In [ ]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    batch_size=batch_size,
    epochs=epochs,
    callbacks=[
        wer,
        ce_der,
        early_stopping,
        model_checkpoint_callback
    ],
)

In [ ]:
# !pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

history_dict = history.history
epochs_range = range(1, len(history_dict["loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Loss
axes[0].plot(epochs_range, history_dict["loss"], marker="o", label="Train Loss")
axes[0].plot(epochs_range, history_dict["val_loss"], marker="o", label="Validation Loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# 2. DER
axes[1].plot(epochs_range, history_dict["der"], marker="o", label="Train DER")
axes[1].plot(epochs_range, history_dict["val_der"], marker="o", label="Validation DER")
axes[1].plot(
    epochs_range,
    history_dict["val_der_case_ending"],
    marker="o",
    label="Case Ending DER"
)
axes[1].set_title("Diacritic Error Rate")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Error Rate")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# 3. WER
if "wer" in history_dict:
    axes[2].plot(epochs_range, history_dict["wer"], marker="o", label="Train WER")

axes[2].plot(epochs_range, history_dict["val_wer"], marker="o", label="Validation WER")
axes[2].set_title("Word Error Rate")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("WER")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.suptitle("Model Training Metrics", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
save_path = f"Mushakkil_model.{model}.keras"
model.save(save_path)
print(f"Saved model to {save_path}")